In [1]:
DRIVE = '/content/drive/MyDrive/aitm-env'
VENV  = '/content/.venv'
PYTHON = f'{VENV}/bin/python'

import subprocess, os

# Kill stuck vLLM
subprocess.run('pkill -f vllm', shell=True)
print('Killed existing vLLM')

# Check GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

# Reinstall vllm fresh via pip (HuggingFace recommended)
!pip install vllm -q
print('vllm installed')

Killed existing vLLM
NVIDIA H100 80GB HBM3, 81559 MiB
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.1/433.1 MB 6.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.3/194.3 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.7/267.7 MB 10.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 181.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 119.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 132.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 662.1/662.1 kB 66.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import subprocess, os, time, urllib.request

DRIVE = '/content/drive/MyDrive/aitm-env'
os.makedirs(f'{DRIVE}/hf_cache', exist_ok=True)
os.environ['HF_TOKEN']               = userdata.get('HF_TOKEN')
os.environ['HUGGING_FACE_HUB_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['HF_HOME']                = f'{DRIVE}/hf_cache'
os.environ['HUGGINGFACE_HUB_CACHE']  = f'{DRIVE}/hf_cache'

MODEL = 'google/gemma-4-31b-it'
LOG   = f'{DRIVE}/vllm.log'
PORT  = 8000

proc = subprocess.Popen(
    ['vllm', 'serve', MODEL,
     '--max-model-len',          '8192',
     '--gpu-memory-utilization', '0.90',
     '--dtype',                  'bfloat16',
     '--port',                   str(PORT),
    ],
    stdout=open(LOG, 'w'),
    stderr=subprocess.STDOUT,
    env={**os.environ},
)
print(f'vLLM PID {proc.pid} starting on H100...')

print('Waiting', end='', flush=True)
for i in range(120):
    try:
        urllib.request.urlopen(f'http://localhost:{PORT}/health')
        print(f'\nvLLM READY after {i*5}s ✓')
        break
    except:
        if i > 0 and i % 6 == 0:
            tail = subprocess.run(f'tail -2 {LOG}', shell=True, capture_output=True, text=True)
            if tail.stdout.strip():
                print(f'\n[{i*5}s] {tail.stdout.strip()}', flush=True)
        print('.', end='', flush=True)
        time.sleep(5)
else:
    print('\nTimeout — last log lines:')
    !tail -30 {LOG}

Mounted at /content/drive
vLLM PID 1469 starting on H100...
Waiting......
[30s] (APIServer pid=1469) INFO 04-23 17:56:49 [utils.py:299] 
(APIServer pid=1469) INFO 04-23 17:56:49 [utils.py:233] non-default args: {'model_tag': 'google/gemma-4-31b-it', 'model': 'google/gemma-4-31b-it', 'dtype': 'bfloat16', 'max_model_len': 8192}
......
[60s] (APIServer pid=1469) INFO 04-23 17:57:07 [vllm.py:790] Asynchronous scheduling is enabled.
(EngineCore pid=1880) INFO 04-23 17:57:28 [core.py:105] Initializing a V1 LLM engine (v0.19.1) with config: model='google/gemma-4-31b-it', speculative_config=None, tokenizer='google/gemma-4-31b-it', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=8192, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None,

In [ ]:
# Model is loaded (58.9 GiB), now doing CUDA graph capture — just wait
import time, urllib.request, subprocess

PORT  = 8000
DRIVE = '/content/drive/MyDrive/aitm-env'
LOG   = f'{DRIVE}/vllm.log'

print('Waiting for CUDA graph capture to finish...', end='', flush=True)
for i in range(120):
    try:
        urllib.request.urlopen(f'http://localhost:{PORT}/health')
        print(f'\nvLLM READY ✓ (total ~{570 + i*5}s from start)')
        break
    except:
        if i % 6 == 0:
            tail = subprocess.run(f'tail -2 {LOG}', shell=True, capture_output=True, text=True)
            if tail.stdout.strip():
                print(f'\n[+{i*5}s] {tail.stdout.strip()}', flush=True)
        print('.', end='', flush=True)
        time.sleep(5)
else:
    print('\nStill waiting — last 10 log lines:')
    !tail -10 {LOG}

Waiting for CUDA graph capture to finish...
[+0s] (EngineCore pid=1880) INFO 04-23 18:16:10 [gpu_model_runner.py:4820] Model loading took 58.9 GiB memory and 1114.922961 seconds
(EngineCore pid=1880) INFO 04-23 18:16:10 [gpu_model_runner.py:5753] Encoder cache will be initialized with a budget of 8192 tokens, and profiled with 3 video items of the maximum feature size.
......
[+30s] (EngineCore pid=1880) INFO 04-23 18:24:05 [backends.py:1051] Using cache directory: /root/.cache/vllm/torch_compile_cache/9f05ecb58c/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=1880) INFO 04-23 18:24:05 [backends.py:1111] Dynamo bytecode transform time: 13.32 s
......
[+60s] (EngineCore pid=1880) INFO 04-23 18:24:05 [backends.py:1111] Dynamo bytecode transform time: 13.32 s
(EngineCore pid=1880) INFO 04-23 18:25:27 [backends.py:372] Cache the graph of compile range (1, 8192) for later use
......
[+90s] (EngineCore pid=1880) INFO 04-23 18:25:27 [backends.py:372] Cache the graph of compile rang

In [ ]:
import subprocess, os, time, urllib.request
from google.colab import drive, userdata

# Ensure Drive is mounted
try:
    drive.mount('/content/drive')
except:
    pass

DRIVE = '/content/drive/MyDrive/aitm-env'
os.makedirs(f'{DRIVE}/hf_cache', exist_ok=True)
os.environ['HF_TOKEN']               = userdata.get('HF_TOKEN')
os.environ['HUGGING_FACE_HUB_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['HF_HOME']                = f'{DRIVE}/hf_cache'
os.environ['HUGGINGFACE_HUB_CACHE']  = f'{DRIVE}/hf_cache'

MODEL = 'google/gemma-4-31b-it'
PORT  = 8000
LOG   = f'{DRIVE}/vllm.log'

# Check if already running
ps = subprocess.run('ps aux | grep "vllm serve" | grep -v grep', shell=True, capture_output=True, text=True)
if ps.stdout.strip():
    print('vLLM already running:', ps.stdout.strip()[:100])
else:
    print('vLLM not running — restarting...')
    subprocess.run('pkill -f vllm', shell=True)
    time.sleep(2)
    proc = subprocess.Popen(
        ['vllm', 'serve', MODEL,
         '--max-model-len', '8192',
         '--gpu-memory-utilization', '0.90',
         '--dtype', 'bfloat16',
         '--port', str(PORT),
        ],
        stdout=open(LOG, 'w'),
        stderr=subprocess.STDOUT,
        env={**os.environ},
    )
    print(f'vLLM PID {proc.pid} restarted')

# Poll health
print('Waiting for vLLM', end='', flush=True)
for i in range(120):
    try:
        urllib.request.urlopen(f'http://localhost:{PORT}/health')
        print(f'\nvLLM READY ✓')
        break
    except:
        print('.', end='', flush=True)
        time.sleep(5)
else:
    print('\nTimeout — last log lines:')
    !tail -10 {DRIVE}/vllm.log

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
vLLM not running — restarting...
vLLM PID 20694 restarted
Waiting for vLLM........................................................................................................................
Timeout — last log lines:
(EngineCore pid=20883) INFO 04-23 19:10:39 [parallel_state.py:1400] world_size=1 rank=0 local_rank=0 distributed_init_method=tcp://172.28.0.12:58515 backend=nccl
(EngineCore pid=20883) INFO 04-23 19:10:39 [parallel_state.py:1716] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, PCP rank 0, TP rank 0, EP rank N/A, EPLB rank N/A
(EngineCore pid=20883) INFO 04-23 19:10:40 [gpu_model_runner.py:4735] Starting to load model google/gemma-4-31b-it...
(EngineCore pid=20883) INFO 04-23 19:10:40 [vllm.py:790] Asynchronous scheduling is enabled.
(EngineCore pid=20883) INFO 04-23 19:10:40 [cuda.py:274] Using AttentionBackendEnum.TRITON_ATTN bac

# AITM Red-Teaming — Colab vLLM Setup (Reusable via Google Drive)

**Model:** `google/gemma-4-31b-it` served by vLLM on `localhost:8000`  
**Persistence:** uv venv + model weights cached on Drive → fast reconnect every session

| Session | Time |
|---|---|
| First run (downloads 60GB model) | ~30-45 min |
| Reconnect (loads from Drive) | ~3-5 min |

In [ ]:
# ── CELL 1: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/aitm-env'
!mkdir -p {DRIVE}/hf_cache
print('Drive mounted. Base path:', DRIVE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted. Base path: /content/drive/MyDrive/aitm-env


In [ ]:
# ── CELL 2: Install / restore uv from Drive ───────────────────────────────────
import os, subprocess

UV_CACHED = f'{DRIVE}/uv'
UV_BIN    = '/usr/local/bin/uv'

if os.path.exists(UV_CACHED):
    subprocess.run(f'cp {UV_CACHED} {UV_BIN} && chmod +x {UV_BIN}', shell=True)
    print('uv restored from Drive (no download needed)')
else:
    subprocess.run('curl -LsSf https://astral.sh/uv/install.sh | sh', shell=True)
    result = subprocess.run(
        'find ~/.local/bin /root/.local/bin -name uv 2>/dev/null | head -1',
        shell=True, capture_output=True, text=True
    )
    uv_path = result.stdout.strip()
    subprocess.run(f'cp {uv_path} {UV_CACHED} && cp {uv_path} {UV_BIN} && chmod +x {UV_BIN}', shell=True)
    print(f'uv installed and cached to Drive')

!uv --version

uv installed and cached to Drive
uv 0.11.7 (x86_64-unknown-linux-gnu)


In [ ]:
# ── CELL 3: Install vllm via pip (fast, ~2 min, always fresh) ─────────────────
# Skips uv venv — pip installs into system Python which `vllm serve` uses
import sys, subprocess

!pip install vllm -q
print('vllm installed ✓')

result = subprocess.run(['vllm', '--version'], capture_output=True, text=True)
print('vllm version:', result.stdout.strip() or result.stderr.strip())

PYTHON = sys.executable
print('Python:', PYTHON)

venv already on local SSD ✓
Python 3.12.13


In [ ]:
# ── CELL 4: HF token + cache path ────────────────────────────────────────────
import os
from google.colab import userdata


HF_TOKEN = userdata.get('HF_TOKEN')  # paste your token: huggingface.co/settings/tokens (needs Gemma access)

os.environ['HF_TOKEN']               = HF_TOKEN
os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN
os.environ['HF_HOME']                = f'{DRIVE}/hf_cache'
os.environ['HUGGINGFACE_HUB_CACHE']  = f'{DRIVE}/hf_cache'

print('HF cache →', os.environ['HF_HOME'])
print('Token set' if HF_TOKEN else 'WARNING: HF_TOKEN is empty — set it before launching vLLM!')

HF cache → /content/drive/MyDrive/aitm-env/hf_cache
Token set


In [ ]:
# ── CELL 5: Launch vLLM on localhost:8000 ────────────────────────────────────
# First run: downloads gemma-4-31b-it weights to Drive (~60 GB, ~30 min)
# Reconnect: loads from Drive cache (~3-5 min)
import subprocess, time, urllib.request

MODEL = 'google/gemma-4-31b-it'
PORT  = 8000
LOG   = f'{DRIVE}/vllm.log'

proc = subprocess.Popen(
    [
        PYTHON, '-m', 'vllm.entrypoints.openai.api_server',
        '--model',                  MODEL,
        '--max-model-len',          '8192',
        '--gpu-memory-utilization', '0.90',
        '--tensor-parallel-size',   '1',
        '--port',                   str(PORT),
    ],
    stdout=open(LOG, 'w'),
    stderr=subprocess.STDOUT,
    env={**os.environ},
)
print(f'vLLM PID {proc.pid} starting...')
print(f'Logs → {LOG}  (tail it in Cell 5b if needed)')

vLLM PID 4755 starting...
Logs → /content/drive/MyDrive/aitm-env/vllm.log  (tail it in Cell 5b if needed)


In [ ]:
# ── CELL 5b: Poll until vLLM is ready ────────────────────────────────────────
import time, urllib.request, subprocess

PORT = 8000
print('Waiting for vLLM...', end='', flush=True)
for i in range(180):
    try:
        urllib.request.urlopen(f'http://localhost:{PORT}/health')
        print(f'\nvLLM READY after {i*5}s ✓')
        break
    except:
        if i % 6 == 0:  # every 30s print log tail
            log = subprocess.run(f'tail -3 {DRIVE}/vllm.log', shell=True, capture_output=True, text=True)
            if log.stdout.strip():
                print(f'\n[{i*5}s] {log.stdout.strip()}', flush=True)
        print('.', end='', flush=True)
        time.sleep(5)
else:
    print('\nTimeout — full log:')
    !tail -30 {DRIVE}/vllm.log

Waiting for vLLM...
[0s] (APIServer pid=4755) INFO 04-23 07:02:41 [utils.py:299]    ▀▀  ▀▀▀▀▀ ▀▀▀▀▀ ▀     ▀
(APIServer pid=4755) INFO 04-23 07:02:41 [utils.py:299] 
(APIServer pid=4755) INFO 04-23 07:02:41 [utils.py:233] non-default args: {'model': 'google/gemma-4-31b-it', 'max_model_len': 8192}
......
[30s] (EngineCore pid=4916) INFO 04-23 07:03:04 [cuda.py:274] Using AttentionBackendEnum.TRITON_ATTN backend.
(EngineCore pid=4916) <frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
(EngineCore pid=4916) <frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.
......
[60s] (EngineCore pid=4916) INFO 04-23 07:03:04 [cuda.py:274] Using AttentionBackendEnum.TRITON_ATTN backend.
(EngineCore pid=4916) <fro

In [ ]:
# ── CELL 6: Quick smoke test — one inference call ─────────────────────────────
from openai import OpenAI

client = OpenAI(base_url=f'http://localhost:{PORT}/v1', api_key='EMPTY')

resp = client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'user', 'content': 'Say hello in one sentence.'}],
    max_tokens=64,
)
print('Model:', MODEL)
print('Response:', resp.choices[0].message.content.strip())
print('Tokens used:', resp.usage.total_tokens)

In [ ]:
# ── CELL 7: Clone / update repo + install project deps ────────────────────────
import os

REPO     = 'https://github.com/highphysicist/aitm-red-teaming-mas.git'
BRANCH   = 'Gemma-colab-judge-task'
REPO_DIR = '/content/aitm'

if not os.path.exists(REPO_DIR):
    !git clone --branch {BRANCH} {REPO} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} pull origin {BRANCH}

%cd {REPO_DIR}

# Install project deps into Drive venv (cached — skips already-installed packages)
!uv pip install --python {PYTHON} -r requirements.txt -q
print('Repo ready at', REPO_DIR)

In [ ]:
# ── CELL 8: Run benchmark ─────────────────────────────────────────────────────
# config.py on Gemma-colab-judge-task already points to localhost:8000
# Edit args as needed

!{PYTHON} benchmark.py \
    --adapter autogen \
    --topo chain \
    --dataset mbpp \
    --n_samples 5 \
    --attack_type targeted \
    --k 1 \
    --save_log

In [ ]:
import subprocess, os
gpu = subprocess.run('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader', shell=True, capture_output=True, text=True)
print('GPU:', gpu.stdout.strip())
print('Drive mounted:', os.path.exists('/content/drive/MyDrive/aitm-env'))
print('venv exists:', os.path.exists('/content/.venv/bin/python'))
print('vllm in PATH:', subprocess.run('which vllm', shell=True, capture_output=True, text=True).stdout.strip())
